# Exercise 2: Read a return by its paths

**Goal:** After this exercise, you can find three Form 1040 fields by their paths with ElementTree, and label each field.

**Scenario:** Week 3's case-selection work starts from return files like this sample. Codex writes the path to three fields of its Form 1040, and you sort each field as made-up test value, protected data or instructor-only label. Then you break the file on purpose and read what the parser says.

**Coding agent:** yes. You use Codex in a terminal.

**Where you work:** the folder `week-02/day-03/exercise-02-read-a-return-by-its-paths/lab` in the course folder. It is a Git repository of its own. You run the cells of this notebook, edit one file in the editor, and run some commands in a terminal.

**Done when:** cell 4 prints `Done:`.

**Optional part:** in step 4, also delete a different closing tag in a second copy, and compare the line the parser names with the line you changed.

**What you have:** in `lab`:

- `returns/sample-return.xml`, a tax return written as XML. The course made it up: no real taxpayer and no real return.
- `paths.csv`, three rows, one for each Form 1040 field, with the columns `field`, `path` and `label`. The `path` and `label` columns are empty.
- `AGENTS.md`, which tells Codex how to write a path and never to print a value of a field that identifies a person.

**Start from:** step 1. To start again, run the reset command of step 1 again.

### Step 1. See the start

Open a terminal (**Terminal > New Terminal**). It opens in the course folder. Run:

```sh
source .venv/bin/activate
python3 reset.py exercise-02-read-a-return-by-its-paths
```

Then run cell 1. It prints the root element's tag, the tags of its two children, and the three rows of `paths.csv`. You should see `{http://www.irs.gov/efile}Return` first: ElementTree writes a tag in a namespace as `{uri}tag`.

In [ ]:
# 1. See the start. You do not need to read this code.
import csv
import xml.etree.ElementTree as ET
from pathlib import Path

lab = Path(_dh[0]) / "lab"
if not (lab / ".git").is_dir():
    print("lab is not there yet. Run the two commands of step 1 in a terminal.")
else:
    root = ET.parse(lab / "returns" / "sample-return.xml").getroot()
    print("Root element:", root.tag)
    for child in root:
        print("  Its child:", child.tag)
    with open(lab / "paths.csv", newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            print(row)

### Step 2. Have Codex write the three paths

Open `lab/returns/sample-return.xml` in the editor (in the Explorer: `week-02 > day-03 > exercise-02-read-a-return-by-its-paths > lab > returns`). Find `ReturnData`, then `IRS1040` inside it: the Form 1040. Find the three fields that `paths.csv` names.

In the terminal, run these two commands:

```sh
cd week-02/day-03/exercise-02-read-a-return-by-its-paths/lab
codex
```

If Codex asks if you trust this folder, answer yes. Write your request in your own words. It must say these things:

| What your request says | Why |
|---|---|
| Write the path of each of the three fields of `paths.csv` in its `path` column, for `returns/sample-return.xml` | "A path names each element on the way down, one step per element." |
| Each path starts at the root element and names one element per step, with the prefixes `efile:` and `course:` that `AGENTS.md` gives | "Every element of the return is in a namespace, so the path has a prefix on every step." |
| No `//` | `//` "Selects all subelements, on all levels beneath the current element." "The sample has two elements named `WagesAmt`: one in the Form 1040 and one in the W-2." |
| Leave the `label` column empty | You sort the fields yourself, in step 3. |
| Print no value: only the tag each path reaches | One of the fields is an SSN. |

<details><summary>Stuck? One way to say it</summary>

`In paths.csv, write the path of each of the three fields in its path column, for returns/sample-return.xml. Each path starts at the root element Return and names one element per step, with the prefixes efile: and course: from AGENTS.md, and no //. Check each path with ElementTree's find, but print only the tag it reaches, never a value. Leave the label column empty.`

</details>

When Codex reports its change, type `/quit`. Then press Enter. Run cell 2: for each row, it prints the tag the path reaches. A row with no label yet shows no value.

In [ ]:
# 2. Follow each path. You do not need to read this code.
import csv
import xml.etree.ElementTree as ET
from pathlib import Path

NS = {"efile": "http://www.irs.gov/efile", "course": "urn:course:instructor"}
lab = Path(_dh[0]) / "lab"
root = ET.parse(lab / "returns" / "sample-return.xml").getroot()
with open(lab / "paths.csv", newline="", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        path, label = (row["path"] or "").strip(), (row["label"] or "").strip()
        found = root.find(path, NS) if path else None
        if found is None:
            print(f"{row['field']}: {'no path yet' if not path else 'the path finds nothing'}")
        elif not label:
            print(f"{row['field']}: reaches {found.tag}. Label it to see its value.")
        elif label == "protected data":
            print(f"{row['field']}: reaches {found.tag}. Protected data: its value is not printed.")
        else:
            print(f"{row['field']}: reaches {found.tag}, value {found.text!r} ({label})")

### Step 3. Label each field

Open `lab/paths.csv` in the editor. Write one label in the `label` column of each row, by this rule. Ask the three questions in order, and stop at the first yes:

| Question | Yes: the label |
|---|---|
| Did the course add the field for the trainer, in its own namespace, `course:`? No real return holds it. | `instructor-only label` |
| Does the field identify a person: an SSN, a name, an address? "Then it is protected data, even when the value is made up, because the same field on a real return is taxpayer data." | `protected data` |
| Neither: an amount, a code or a date the course made up. | `made-up test value` |

Save the file. Run cell 2 again. You should see a value for two rows, and for the protected one, `its value is not printed`.

### Step 4. Break the file, and read the parser's error

In the same terminal, in `lab`, make a copy of the return:

```sh
cp returns/sample-return.xml broken.xml
```

Open `lab/broken.xml` in the editor. Inside `IRS1040`, delete one closing tag, for example `</WagesAmt>`, and save. The file is now not well-formed: an element that was opened is never closed.

Run cell 3. It parses `broken.xml` and prints the parser's error. Make a new file, `lab/parse-error.txt`, and copy the line that cell 3 prints into it. Save it.

Look at the line number in the error. Then look at the line you changed in `broken.xml`.

In [ ]:
# 3. Parse the broken copy. You do not need to read this code.
import xml.etree.ElementTree as ET
from pathlib import Path

broken = Path(_dh[0]) / "lab" / "broken.xml"
if not broken.is_file():
    print("lab/broken.xml is not there yet. Make the copy in step 4.")
else:
    try:
        ET.parse(broken)
        print("broken.xml is still well-formed. Delete one closing tag and save.")
    except ET.ParseError as error:
        print(error)

### Step 5. Check

Run cell 4. It prints `Done:` when these things are true:

- Each path in `paths.csv` reaches its field from the root element, with no `//`.
- Each field has the label the rule of step 3 gives it.
- `lab/broken.xml` is not well-formed, and `lab/parse-error.txt` holds the line the parser names.

Otherwise it prints `Not yet:` and each thing still to do. You do not need to read its code.

In [ ]:
# 4. Check. You do not need to read this code.
%run "{_dh[0]}/check.py"

### Afterwards, together

1. "A path with `//` finds both, and nothing in it says which form you mean." Week 3 reads hundreds of returns. On which return would `.//efile:WagesAmt` give a wrong wage?
2. "The sorting is about the field, not this file's values." Which field of a file your team receives today would you label protected data, and why?
3. "Read upward from the reported line to the element that was never closed." In a return of 2,000 lines, how would you find the fault fast?

<details><summary>Answer</summary>

The paths, from the root `Return`, one step per element:

- wages on Form 1040: `efile:ReturnData/efile:IRS1040/efile:WagesAmt`, a made-up test value.
- the dependent's SSN on Form 1040: `efile:ReturnData/efile:IRS1040/efile:DependentDetail/efile:DependentSSN`, protected data: it identifies a person, even made up.
- the review label on Form 1040: `efile:ReturnData/efile:IRS1040/course:ReviewLabel`, an instructor-only label: the course added it, in its own namespace, so its step takes `course:`.

The parse error: deleting `</WagesAmt>` gives `mismatched tag: line 35, column 6`. Line 35 is `</IRS1040>`, the first closing tag that does not match the open `WagesAmt`. The parser names where it noticed the fault, at or after the line you changed.

The paths and the error are in [`answers/exercise-02-read-a-return-by-its-paths`](../answers/exercise-02-read-a-return-by-its-paths/). Each reason, and where each quoted line comes from, is in [`decisions.md`](../answers/exercise-02-read-a-return-by-its-paths/decisions.md).

</details>